# Reduce file size

In [ ]:
import pandas as pd

# 1. Load dataset (only the columns we actually need)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Define what a "hit" is (e.g. popularity over 75)
threshold = 75
df['is_hit'] = (df['track_popularity'] >= threshold).astype(int)

# 3. Split the dataset
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

print(f"Number of Hits: {len(hits)}")
print(f"Number of Non-Hits before reduction: {len(non_hits)}")

# 4. Balancing & Reduction
# We take ALL hits and exactly as many random non-hits
non_hits_sampled = non_hits.sample(n=len(hits), random_state=42)

# 5. Merge into final training set
df_final = pd.concat([hits, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

print(f"Final dataset size: {len(df_final)}")

In [ ]:
import pandas as pd

# 1. Load dataset (only the columns we actually need)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Define what a "hit" is (e.g. popularity over 60, the new threshold)
new_threshold = 50 # Using the new threshold proposed earlier
df['is_hit'] = (df['track_popularity'] >= new_threshold).astype(int)

# 3. Split the dataset
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

print(f"Number of hits with new threshold ({new_threshold}): {len(hits)}")
print(f"Number of Non-Hits before reduction: {len(non_hits)}")

# 4. Balancing & Reduction
# We take ALL hits and as many random non-hits so that the final dataset has 10,000 entries
# The number of non-hits to sample is 10000 - the number of hits
non_hits_to_sample = max(0, 10000 - len(hits)) # Ensure non_hits_to_sample is not negative
non_hits_sampled = non_hits.sample(n=non_hits_to_sample, random_state=42)

# 5. Merge into final training set
df_final = pd.concat([hits, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

print(f"Final dataset size: {len(df_final)}")

# 6. Save final dataset
df_final.to_csv('../data/final_processed_dataset.csv', index=False)
print("Final dataset saved as '../data/final_processed_dataset.csv'.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the 'track_popularity' column
track_popularity = pd.read_csv('../data/master_dataset_enriched.csv', usecols=['track_popularity'])

# 2. Create a histogram to visualize its distribution
plt.figure(figsize=(10, 6))
sns.histplot(track_popularity['track_popularity'], bins=50, kde=True)

# 3. Add appropriate labels and title
plt.title('Distribution of Track Popularity')
plt.xlabel('Track Popularity')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)

# 4. Display the plot
plt.show()

In [ ]:
import pandas as pd

# 1. Load the full dataset (only relevant columns for defining hits)
cols_to_keep = ['track_popularity']
df_full = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Define the new 'hit' threshold
new_threshold = 50
df_full['is_hit'] = (df_full['track_popularity'] >= new_threshold).astype(int)

# 3. Recalculate the number of hits and non-hits
new_hits_count = df_full[df_full['is_hit'] == 1].shape[0]
new_non_hits_count = df_full[df_full['is_hit'] == 0].shape[0]

print(f"New Threshold for Hits: {new_threshold}")
print(f"Number of Hits with new threshold: {new_hits_count}")
print(f"Number of Non-Hits with new threshold: {new_non_hits_count}")

In [ ]:
import pandas as pd

# 1. Load columns (now including release_year and speechiness for filtering)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness', 'release_year']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# --- NEW: FILTER STEP ---
# Only modern songs (from 2000)
df = df[df['release_year'] >= 2000]

# Only real songs (no podcasts/audiobooks)
df = df[df['speechiness'] < 0.5]

# No pure instrumental tracks (optional, depending on goal)
df = df[df['instrumentalness'] < 0.5]
# ---------------------------

# 2. Define hit
new_threshold = 50
df['is_hit'] = (df['track_popularity'] >= new_threshold).astype(int)

# 3. Split the dataset
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

# 4. Clean 50/50 balancing for 10,000 entries
# We take 5,000 hits and 5,000 non-hits
n_per_class = 5000

if len(hits) < n_per_class:
    n_per_class = len(hits) # If we have fewer than 5,000 hits after filtering

hits_sampled = hits.sample(n=n_per_class, random_state=42)
non_hits_sampled = non_hits.sample(n=n_per_class, random_state=42)

# 5. Merge & Shuffle
df_final = pd.concat([hits_sampled, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

# Release year is no longer needed for training
df_final = df_final.drop(columns=['release_year'])

print(f"Final size: {len(df_final)} (50% hits, 50% non-hits)")
df_final.to_csv('../data/final_processed_dataset_V2.csv', index=False)